## Label different applications by integer

In [ ]:
APP_TO_LABEL ={'xyz':0,
               'fh':1,
               'ti':2,
               'h2':3,
               'hehp':4,
               'nh3':5,
               'random':6,}


LABEL_TO_APP = {0: 'Heisenberg XYZ', 
                1: 'Fermi-Hubbard',
                2: 'Transverse Ising',
                3: 'H2',
                4: 'HeH+',
                5: 'NH3',
                6: 'Random VQE'}


FIG_SIZE = (6, 6)

In [ ]:
from pathlib import Path
import h5py
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_distances 
from sklearn.manifold import TSNE
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib
import os
from tqdm.notebook import tqdm
import seaborn as sns
from scipy.stats import gaussian_kde
from matplotlib.cm import get_cmap







def get_padding_len(dir_path):
    app_dir = Path(dir_path)

    apps = list(app_dir.glob('*.h5'))


    apps = [str(f) for f in apps if f.is_file() and 'backup' not in str(f)]

    max_len = 0
    for app in apps:
        with h5py.File(app, 'r') as f:
            assert 'opt_params' in f.keys(), f"opt_params not found in {app}"
            opt_param_group = f['opt_params']
            assert 'sample_0' in opt_param_group.keys(), f"sample_0 not found in {app}"
            opt_param_array = opt_param_group['sample_0'][...]
            flat = opt_param_array.flatten()
            if len(flat) > max_len:
                max_len = len(flat)
    # print(f"Max length of model_param array: {max_len}")
    return max_len


def get_app_paths(dir_path):
    from pathlib import Path
    app_dir = Path(dir_path)
    apps = list(app_dir.glob('*.h5'))
    apps = [str(f) for f in apps if f.is_file() and 'backup' not in str(f)]
    return apps



def pad_flatten(opt_param, target_len):
    flat = opt_param.flatten()
    if len(flat) < target_len:
        padding = np.zeros(target_len - len(flat))
        return np.concatenate((flat, padding))
    else:
        return flat[:target_len]
    

def get_app_label(app_name):
    for app, label in APP_TO_LABEL.items():
        if app in app_name:
            return label
    return None



def distance_matrix(params, labels, inter_class_boost = 3.0):

    D = cosine_distances(params)

    labels = np.array(labels)

    if inter_class_boost > 1.0:
        boost_mask = (labels[:, None] != labels[None, :]).astype(float)
        D += (boost_mask * inter_class_boost)

    return D

def _axes_rect(ax, left=0.12, bottom=0.12, width=0.80, height=0.80):
    ax.set_position([left, bottom, width, height])


def _plot(df, color_dict = None, save_path=None):
    fig, ax = plt.subplots(figsize=FIG_SIZE, dpi=400)
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(True, which="major", axis="both", alpha=0.25, linestyle=':')

    for label in sorted(df['labels'].unique()):
        _df = df.query(f"labels=={label}")
        app_name = LABEL_TO_APP[label]
        color = color_dict[app_name]
        ax.scatter(_df.xdata, _df.ydata, color=color, s=9,  alpha=1.0, linewidths=0, rasterized=True)

    _axes_rect(ax)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, format='pdf', bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    else:
        plt.show()
    plt.close(fig)
    



import matplotlib.patheffects as pe

def _add_qubit_lines(ax, yvals, nqvals, order='asc',
                     colors=((0.10,0.10,0.10,1.0),  # dark gray
                             (1.00,0.40,0.00,1.0),  # orange
                             (0.60,0.30,1.00,1.0)), # purple
                     lw=1.2, label_fs=8, half_width = 0.023):
    """
    Draw short horizontal lines at each subgroup's median ground_energy,
    with a small 'Xq' label (X = n_qubits). Lines sit inside the violin.
    """
    uq = np.unique(nqvals)
    if order == 'asc':
        groups = np.sort(uq)
    elif order == 'freq':
        counts = [(nq, np.sum(nqvals == nq)) for nq in uq]
        groups = [nq for nq,_ in sorted(counts, key=lambda t: (-t[1], t[0]))]
    else:
        groups = uq

    # Center of the (only) violin category on the x-axis (seaborn uses 0,1,...)
    x_center = ax.get_xticks()[0] if len(ax.get_xticks()) > 0 else 0.0

    for i, nq in enumerate(groups[:3]):  # at most 3 groups as you noted
        c = colors[i % len(colors)]
        ymed = np.median(yvals[nqvals == nq])
        x0, x1 = x_center - half_width, x_center + half_width
        ax.plot([x0, x1], [ymed, ymed], color=c, lw=lw, solid_capstyle='round')
        # ax.text(x1 + 0.04, ymed, f'{int(nq)}q', va='center', ha='left',
        #         fontsize=label_fs, color=c)

def violin_plot_single(df, label, save_path=None,
                       line_order='asc',  # or 'freq'
                       line_colors=((0.10,0.10,0.10,1.0),
                                    (1.00,0.40,0.00,1.0),
                                    (0.60,0.30,1.00,1.0)), half_width=0.023):

    _df = df[df["labels"] == label].copy()
    nq_unique = _df['n_qubits'].unique()

    violin_color = (0.5, 0.75, 1.0, 0.6)

    fig, ax = plt.subplots(figsize=(3, 4), dpi=300)
    sns.violinplot(x="labels", y="ground_energy", data=_df, ax=ax,
                   palette=[violin_color], cut=0, inner="box")

    # line span
    x_center = ax.get_xticks()[0] if len(ax.get_xticks()) > 0 else 0.0
    x0, x1 = x_center - half_width, x_center + half_width

    legend_handles = []

    if len(nq_unique) != 1:
        _add_qubit_lines(ax,
            yvals=_df["ground_energy"].to_numpy(),
            nqvals=_df["n_qubits"].to_numpy(),
            order=line_order, colors=line_colors,
            lw=1.25, label_fs=8, half_width=half_width)

        uq = np.unique(_df["n_qubits"].to_numpy())
        if line_order == 'asc':
            groups = np.sort(uq)
        elif line_order == 'freq':
            counts = [(nq, np.sum(_df["n_qubits"].to_numpy() == nq)) for nq in uq]
            groups = [nq for nq,_ in sorted(counts, key=lambda t: (-t[1], t[0]))]
        else:
            groups = uq
        for i, nq in enumerate(groups[:3]):
            legend_handles.append(Line2D([0],[0],
                color=line_colors[i % len(line_colors)], lw=1.25,
                label=f'{int(nq)} qubits'))

    # overall median (same length as subgroup lines)
    overall_med = np.median(_df["ground_energy"].to_numpy())
    stroke = [pe.Stroke(linewidth=2.0+1.2, foreground='black'), pe.Normal()]
    ax.plot([x0, x1], [overall_med, overall_med],
            color='white', lw=2.0, solid_capstyle='round', path_effects=stroke)
    legend_handles.insert(0, Line2D([0],[0], color='white', lw=2.0,
        label='Overall', path_effects=[pe.Stroke(linewidth=3.2, foreground='black'),
                                       pe.Normal()]))
    
    ax.set_xlabel(""); ax.set_xticks([]); ax.set_ylabel("")

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.set_xlim(xmin, xmax + 0.17*(xmax - xmin))
    ax.set_ylim(ymin, ymax)
    legend_labels = [h.get_label() for h in legend_handles]

    ax.legend(handles=legend_handles,
          labels=legend_labels,
          loc='center right',   # inside the frame
          frameon=False,
          fontsize=7,
        bbox_to_anchor=(1.0, 0.58), # keep x=1.0, change y to 0.6 (0.5 is middle)
          handlelength=1.0,
          handletextpad=0.4,
          borderpad=0.4) 
  
    # ----------------------------------------------------------------------

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, format='pdf')   # keep canvas (no tight bbox)
        plt.close(fig)
        print(f"Plot saved to {save_path}")
    else:
        plt.show()

## Read data from h5 files

In [ ]:
data_by_domain = defaultdict(list)
app_dirs = ['qmanybody', 'qchem', 'qasmbench']

for app_dir in app_dirs:
    padding_len = get_padding_len(app_dir)
    print(f"Padding length for {app_dir}: {padding_len}")
    app_paths = get_app_paths(app_dir)

    for app in app_paths:
        with h5py.File(app, 'r') as f:
            assert 'opt_params' in f.keys(), f"opt_params not found in {app}"
            assert 'n_qubits' in f.keys(), f"n_qubits not found in {app}"
            opt_param_group = f['opt_params']
            n_qubits_group = f['n_qubits']
            app_label = get_app_label(app)
            assert app_label is not None, f"Label not found for {app}"
            for key in opt_param_group.keys():
                opt_param = opt_param_group[key][...]
                n_qubits = n_qubits_group[key][...].item()
                padded_opt_param = pad_flatten(opt_param, padding_len)
                data_by_domain[app_dir].append((padded_opt_param, app_label, n_qubits))


## Represent different app by different cololors. 

The map is stored in the dictionary `APP_TO_COLOR` 

In [ ]:

all_apps = []
for domain, samples in data_by_domain.items():
    for _, label, _ in samples:
        all_apps.append(LABEL_TO_APP[label])


all_apps = sorted(set(all_apps))


def distinct_colors(n):
    cmaps = [plt.get_cmap('tab10')]
    bank = []

    for cmap in cmaps:
        bank.extend([cmap(i) for i in range(cmap.N)])
    
    if n < len(bank):
        return bank[:n]
    return [plt.cm.hsv(i/n) for i in range(n)]


APP_TO_COLOR = dict(zip(all_apps, distinct_colors(len(all_apps))))




## Plot Fig. 4 and Fig. 5
For Fermi-Hubbard and Heisenberg XYZ applications. 

In [ ]:
save_dir = './figures/scatter/n_qubits'

qmanybody_data = data_by_domain['qmanybody']

xyz_data = [(p, l, n) for (p, l, n) in qmanybody_data if l == 0]
fh_data = [(p, l, n) for (p, l, n) in qmanybody_data if l == 1]

xyz_by_qubit  = defaultdict(list)

fh_by_qubit  = defaultdict(list)

for (p, l, n) in xyz_data:
    xyz_by_qubit[n].append((p, l))

for (p, l, n) in fh_data:
    fh_by_qubit[n].append((p, l))


for n_qubits, samples in fh_by_qubit.items():
    # Process samples for each number of qubits
    print(f"processing {n_qubits} qubits data")

    params, labels = zip(*samples)
    X = np.stack(params)
    X = StandardScaler().fit_transform(X)
    D = distance_matrix(X, labels, inter_class_boost=0.5)

    tsne = TSNE(n_components=2, random_state=1, perplexity=30, init= 'random', learning_rate='auto', metric='precomputed').fit_transform(D)
    tmp_dict = {'xdata': tsne[:, 0], 'ydata': tsne[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"fh_{n_qubits}_tsne.pdf")
    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)

for n_qubits, samples in fh_by_qubit.items():
    params, labels = zip(*samples)
    X = np.stack(params)
    X = StandardScaler().fit_transform(X)
    D = distance_matrix(X, labels, inter_class_boost=0.5)
    mds = MDS(n_components=2, random_state=30, dissimilarity='precomputed').fit_transform(D)
    tmp_dict = {'xdata': mds[:, 0], 'ydata': mds[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"fh_{n_qubits}_mds.pdf")
    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)

for n_qubits, samples in xyz_by_qubit.items():
    # Process samples for each number of qubits
    print(f"processing {n_qubits} qubits data")

    params, labels = zip(*samples)
    X = np.stack(params)
    X = StandardScaler().fit_transform(X)
    D = distance_matrix(X, labels, inter_class_boost=0.5)

    tsne = TSNE(n_components=2, random_state=1, perplexity=30, init= 'random', learning_rate='auto', metric='precomputed').fit_transform(D)
    tmp_dict = {'xdata': tsne[:, 0], 'ydata': tsne[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"xyz_{n_qubits}_tsne.pdf")
    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)

for n_qubits, samples in xyz_by_qubit.items():
    params, labels = zip(*samples)
    X = np.stack(params)
    X = StandardScaler().fit_transform(X)
    D = distance_matrix(X, labels, inter_class_boost=0.5)
    mds = MDS(n_components=2, random_state=30, dissimilarity='precomputed').fit_transform(D)
    tmp_dict = {'xdata': mds[:, 0], 'ydata': mds[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"xyz_{n_qubits}_mds.pdf")

    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)

## Plot the violin plots in Fig 6

In [ ]:
violin_save_dir = './figures/violin'

gs_energy_by_domain = defaultdict(list)
for app_dir in app_dirs:
    app_paths = get_app_paths(app_dir)
    for app in app_paths:
        with h5py.File(app, 'r') as f: 
            assert 'loss_history' in f.keys(), f"loss_history not found in {app}"
            assert 'n_qubits' in f.keys(), f"n_qubits not found in {app}"
            n_qubits_group = f['n_qubits']
            loss_history_group = f['loss_history']
            app_label = get_app_label(app)
            assert app_label is not None, f"Label not found for {app}"
            for key in loss_history_group.keys():
                loss_history = loss_history_group[key][...]
                n_qubits = n_qubits_group[key][...].item()
                gs_energy = loss_history[-1]
                gs_energy_by_domain[app_dir].append((gs_energy, app_label, n_qubits))





all_nq = sorted(set(n_qubits for _, _, n_qubits in sum(gs_energy_by_domain.values(), [])))
palette = sns.color_palette("tab20", n_colors=len(all_nq))
NQ_TO_COLOR = dict(zip(all_nq, palette))

for domain, samples in gs_energy_by_domain.items():
    X, labels, n_qubits = zip(*samples)
    X = np.array(X)
    labels = np.array(labels)
    n_qubits = np.array(n_qubits)
    unique_labels = np.unique(labels)
    temp_dict = {'labels':labels, 
                 'ground_energy': X,
                 'n_qubits': n_qubits}
    df = pd.DataFrame(temp_dict)

    for label in unique_labels:
        # violin_plot_single(df, label)
        app_name = LABEL_TO_APP[label].replace(" ", "_").lower()
        app_name = app_name.replace('-', '_').replace('+', '')
        save_path = os.path.join(violin_save_dir, f"{app_name}_violin.pdf")

        violin_plot_single(df, label, save_path=save_path, half_width=0.1)





## Plot Fig. 3

In [ ]:
save_dir = './figures/scatter/domain'

for domain, samples in data_by_domain.items():
    print(f"Processing domain: {domain}")
    params, labels, _ = zip(*samples)
    X = np.stack(params)
    X_scaled = StandardScaler().fit_transform(X)
    D = distance_matrix(X_scaled, labels, inter_class_boost=3.0)

    tsne = TSNE(n_components=2, random_state=1, perplexity=30, init= 'random', learning_rate='auto', metric='precomputed').fit_transform(D)
    tmp_dict = {'xdata': tsne[:, 0], 'ydata': tsne[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"{domain}_tsne.pdf")
    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)


for domain, samples in data_by_domain.items():
    print(f"Processing domain: {domain}")
    params, labels, _ = zip(*samples)
    X = np.stack(params)
    X_scaled = StandardScaler().fit_transform(X)
    if domain != 'qasmbench':
        D = distance_matrix(X_scaled, labels, inter_class_boost=3.0)

        mds = MDS(n_components=2, random_state=30, dissimilarity='precomputed').fit_transform(D)
    else:
        D = cosine_distances(X_scaled)
        mds = MDS(n_components=2, random_state=30, dissimilarity='precomputed').fit_transform(D)
    tmp_dict = {'xdata': mds[:, 0], 'ydata': mds[:, 1], 'labels': labels}
    df = pd.DataFrame(tmp_dict)
    save_path = os.path.join(save_dir, f"{domain}_mds.pdf")
    _plot(df, color_dict=APP_TO_COLOR, save_path=save_path)

## Plot the legend strip in Fig. 3

In [ ]:
save_dir = './figures/scatter/domain'

from matplotlib.font_manager import FontProperties
def save_global_legend_strip(app_to_color, out_path,
                             fontsize=14, ncol=6):
    apps_sorted = ['Heisenberg XYZ', 'Fermi-Hubbard', 'Transverse Ising', 'H2', 'HeH+', 'NH3', 'Random VQE']
    print(apps_sorted)
    handles = [Line2D([], [], linestyle='None', marker='o', markersize=16,
                      color=app_to_color[a], label=a) for a in apps_sorted]
    fig = plt.figure(figsize=(16, 0.5))
    bold_font = FontProperties(weight='bold', size=fontsize)

    fig.legend(handles=handles, loc='center', ncol=ncol,
               frameon=False, fontsize=fontsize, )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, format='pdf', bbox_inches='tight', pad_inches=0.02)
    plt.close(fig)
    print(f"Global legend strip saved to {out_path}")
save_global_legend_strip(APP_TO_COLOR,
                         out_path=os.path.join(save_dir, "legend_all_apps.pdf"),
                         fontsize=32, ncol=7)
